<a href="https://colab.research.google.com/github/MightyCrimsonX/Kohya-Anima-Trainer-Mighty/blob/main/MightyCrimson_EasyTrainerScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡Mighty Crimson Lora Easy Scripts Notebook

Este pequeño notebook es para poder realizar entrenamiento de diferentes tipos de loras, ya sea illustrious u anima, con una detallada interfaz para poder tener los mejores ajustes para el dicho entrenamiento.

*Todos los creditos son para Machina ([67372a](https://github.com/67372a)) y su grandioso proyecto [LoRA_Easy_Training_scripts](https://github.com/67372a/LoRA_Easy_Training_scripts)*

In [ ]:
#@markdown # **Instalar dependencias y elegir modelo base de entrenamiento**
from IPython.display import clear_output
from IPython.display import display, HTML
%cd /content
!git clone https://github.com/MightyCrimsonX/LoRA_Easy_Training_Scripts.git
!wget -q https://raw.githubusercontent.com/MightyCrimsonX/Notebook_Scripts/refs/heads/Dev/scripts/download_magic.py
!pip install aria2 gdown
%cd /content/LoRA_Easy_Training_Scripts
!git submodule update --init --recursive
!uv pip install -r requirements.txt
%cd /content/LoRA_Easy_Training_Scripts/backend/sd_scripts
!uv pip install -U typing-extensions~=4.15.0
!uv pip install torch~=2.7.1 torchvision~=0.22.1 --index-url https://download.pytorch.org/whl/cu128
!uv pip install -U --no-deps --force-reinstall git+https://github.com/67372a/RamTorch
!uv pip install -U --no-deps xformers==0.0.31.post1 --index-url https://download.pytorch.org/whl/cu128
!uv pip install -U --no-deps torchao~=0.12.0 --index-url https://download.pytorch.org/whl/cu128
!uv pip install -U -r requirements.txt
!uv pip install -U ../custom_scheduler/.
!uv pip install -U -r ../requirements.txt
!uv pip install -U --force-reinstall --no-deps git+https://github.com/67372a/LyCORIS@dev
import os
import yaml

# 1. Definir la ruta donde accelerate busca la configuración obligatoriamente
config_path = os.path.expanduser("~/.cache/huggingface/accelerate/default_config.yaml")
os.makedirs(os.path.dirname(config_path), exist_ok=True)

# 2. Configuración pre-respondida optimizada para Lightning.ai / Colab (1 GPU)
accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "NO",
    "downcast_bf16": "no",
    "gpu_ids": "all",
    "machine_rank": 0,
    "main_training_function": "main",
    "mixed_precision": "fp16",  # Cambia a "bf16" si usas GPUs de gama alta (L4, A10G, A100)
    "num_machines": 1,
    "num_processes": 1,
    "rdzv_backend": "static",
    "same_network": True,
    "tpu_env": [],
    "tpu_use_cluster": False,
    "tpu_use_sudo": False,
    "use_cpu": False
}

# 3. Guardar el archivo YAML en el sistema de manera silenciosa
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(accelerate_config, f, default_flow_style=False)
clear_output()
import os
import download_magic

models_dir = "/content/models"
os.makedirs(models_dir , exist_ok=True)
os.chdir(models_dir)
training_model = "Anima v1 Base" # @param  ["Anima v1 Base", "illustrious0.1", "illustrious2.0"]
vae_model = "Anima VAE"  #@param ["SDXL VAE", "VAE FP16 Fix", "Anima VAE"]


if "Anima v1 Base" in training_model:
  model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors"
  %download {model_url}
elif "illustrious0.1" in training_model:
  model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-xl-early-release-v0/resolve/main/Illustrious-XL-v0.1.safetensors"
  %download {model_url}
elif "illustrious2.0" in training_model:
  model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-XL-v2.0/resolve/main/Illustrious-XL-v2.0.safetensors"
  %download {model_url}

if "SDXL VAE" in vae_model:
  vae_url = "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors"
  %download {vae_url} -o sdxl_vae.safetensors
elif "VAE FP16 Fix" in vae_model:
  vae_url = "https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors"
  %download {vae_url} -o sdxl_vae.safetensors
elif "Anima VAE" in vae_model:
  vae_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
  %download {vae_url}
if "Anima v1 Base" in training_model:
  textenc= "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
  %download {textenc}


#clear_output()
display(HTML(f"<h1 style='color: yellow;'>Instalación completada</h1>"))



###<font color="blue"> **Elija el directorio en donde estarán alojados el dataset y el ouput resultante.**

In [ ]:
from IPython.display import display, HTML
import os

Project_structure = "Drive" #@param  ["Drive", "Colab"]
Project_name = "prueba" #@param {type:"string"}

if "Drive" in Project_structure:
  from google.colab import drive
  print("📂 Conectando a Google Drive...")
  drive.mount('/content/drive')
  drive_projects = "/content/drive/MyDrive/loras"
  project_path = os.path.join(drive_projects, Project_name)
  dataset_path = os.path.join(project_path, "dataset")
  output_path = os.path.join(project_path, "output")
  os.makedirs(dataset_path, exist_ok=True)
  os.makedirs(output_path, exist_ok=True)
  display(HTML(f"<h2 style='color: white;'><i>Conexion a Drive Realizada, Usa este path dentro de la ui:</h1>"))
  display(HTML(f"<h2 style='color: green;'>Dataset path: {dataset_path}</h1>"))
  display(HTML(f"<h2 style='color: green;'>output path: {output_path}</h1>"))

elif "Colab" in Project_structure:
  colab_projects = "/content/lora"
  output_path = "/content/lora_output"
  project_path = os.path.join(colab_projects, Project_name)
  dataset_path = os.path.join(project_path, "dataset")
  os.makedirs(output_path, exist_ok=True)
  os.makedirs(dataset_path, exist_ok=True)
  display(HTML(f"<h2 style='color: white; '><i>Carpeta creadas dentro de colab, sube el dataset manualmente y usa este path dentro de la ui:</h1>"))
  display(HTML(f"<h1 style='color: green'>Dataset path: {dataset_path}</h1>"))
  display(HTML(f"<h1 style='color: green;'>Output path: {output_path}</h1>"))




📂 Conectando a Google Drive...
Mounted at /content/drive


### <font color="purple">**Modifíque y use esta celda si quiere generar imagenes de prueba a medida que se este entrenando el lora.**</font>
<font color="red">**inestable al entrenar con illustrious!**</font>

In [ ]:
import os, toml
from IPython.display import display, HTML
sample_prompt_dir = "/content/samplefile"
os.makedirs(sample_prompt_dir , exist_ok=True)
sample_prompt_file = os.path.join(sample_prompt_dir, "sample_prompts.txt")
#@markdown Genera una muestra antes de iniciar el entrenamiento.
#@markdown **Prompt positivo** para las imágenes de muestra.
sample_positive_prompt = "masterpiece, best quality" #@param {type:"string"}
#@markdown **Prompt negativo** para las imágenes de muestra.
sample_negative_prompt = "low quality, worst quality" #@param {type:"string"}
#@markdown Resolución de las imágenes de muestra.
sample_resolution = "768x1344" #@param ["832x1216", "1216x832", "768x1344", "1344x768", "768x1024", "1024x768", "1024x1024", "896x1152", "1152x896"]
#@markdown Semilla para la generación (se usará la misma en todas las muestras).
sample_seed = 5000 #@param {type:"number"}
#@markdown CFG Scale para la generación de muestras.
sample_cfg_scale = 4.5 #@param {type:"number"}
#@markdown Número de pasos de generación para las muestras.
sample_steps = 23 #@param {type:"number"}
#@markdown Sampler/Scheduler para la generación de muestras.


sample_w, sample_h = sample_resolution.split("x")
with open(sample_prompt_file, "w") as spf:
  prompt_line = f"{sample_positive_prompt} --n {sample_negative_prompt} --w {sample_w} --h {sample_h} --d {int(sample_seed)} --l {sample_cfg_scale} --s {int(sample_steps)}"
  spf.write(prompt_line + "\n")
display(HTML(f"<h2 style='color: white; '><i>Sample generado, usa el siguiente path dentro de la ui:</h1>"))
display(HTML(f"<h1 style='color: green'>/content/samplefile/sample_prompts.txt</h1>"))


<font color="cyan"> **Inície esta celda y espere a que aparezca el link de cloudflare el cual ya estara listo para entrenar.**

In [ ]:
%cd /content/LoRA_Easy_Training_Scripts/backend

#@markdown #**Iniciar Backend**

import os
import time
import re
import json
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
os.environ.setdefault("ACCELERATE_DISABLE_RICH_PROGRESS", "1")
os.environ["RICH_TRACKING"] = "0"
os.environ["DISABLE_TELEMETRY"] = "1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 2. Indica al sistema que la terminal no soporta interactividad avanzada ni colores complejos
os.environ["TERM"] = "dumb"

# 3. Evita que librerías como 'huggingface' o 'accelerate' saturen la celda con barras de progreso visuales
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# Descargar gradio-tunnel.py si no existe
if not os.path.exists("/content/LoRA_Easy_Training_Scripts/backend/gradio-tunnel.py"):
    !wget -q https://huggingface.co/datasets/Mightys/Notebook_Scripts/resolve/main/scripts/gradio-tunnel.py -O /content/LoRA_Easy_Training_Scripts/backend/gradio-tunnel.py

# ==========================================
# 2. Dependencias
# ==========================================
!pip install -q requests
# 2. Guardar el archivo usando Python puro
config_data = {
    "remote": True,
    "port": 8000
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2)


# ==========================================
# 3. Iniciar túnel Gradio (puerto 8000)
# ==========================================
print("\n🚀 Iniciando túnel Gradio en puerto 8000...")

!rm -f /tmp/gradio_swarm.log
!python /content/LoRA_Easy_Training_Scripts/backend/gradio-tunnel.py 8000 > /tmp/gradio_kohya.log 2>&1 &

# Esperar y leer URL del archivo
url_gradio = None
for intento in range(20):
    time.sleep(1)
    try:
        with open('/tmp/gradio_kohya.log', 'r') as f:
            contenido = f.read()
            match = re.search(r'https://[\w-]+\.gradio\.live', contenido)
            if match:
                url_gradio = match.group(0)
                break
    except:
        pass

if url_gradio:
    print(f"\n{'='*60}")
    print(f"🌐 URL PÚBLICA DE GRADIO: {url_gradio}")
    print(f"   Espere a que inicie correctamente el backend...")
    print(f"{'='*60}\n")
else:
    print("⚠️  URL no detectada, continuando...")
    print("   Log del túnel: !cat /tmp/gradio_kohya.log")

!PYTHONWARNINGS="ignore" TF_CPP_MIN_LOG_LEVEL=3 python main.py


In [ ]:
#@markdown # **📦Descargar carpeta outputs (Solo si se usó Project_structure = Colab)**
import shutil
from google.colab import files

folder_path = "/content/lora_output"
zip_path = "/content/lora_Output.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)
files.download(zip_path)
